## This is the code to train the model and acquire influence for Neighbour Influence 1 Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 160 features in total. The separation is set to 5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 8000, and the test size is 500. The number of features is set to 10. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, this is exactly the same code as the base code. What you expect here is to acquire the estimation result for a certrain experiment setting. Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Training Sample Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **For Neighbour Influence Experiment, you don't need to run several times. But for multi-verification, you can still do the following steps** -> Change the Training Sample Size and Repeat all the process -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [1]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [3]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [4]:
import random
from keras.optimizers import SGD

In [5]:
from sklearn.datasets import make_classification

In [6]:
import time

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The only thing you might need to change in this code is changing the train size. In this experiment, all the other things are fixed, but the number of training samples might be changed to do multi-verification experiments.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. Sep to make sure the dataset is distinguishable.

In [7]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [8]:
df = pd.read_csv("diamonds.csv")

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0           60.599998            60.0        6

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [9]:
median_price = df["price"].median()
df["label"] = (df["price"] > median_price).astype(int)
df = df.drop(columns=['price'])

In [10]:
df['id'] = np.arange(1, len(df) + 1)
print(df)

       features/carat  features/clarity  features/color  features/cut  \
0                1.26                 2               4             2   
1                0.80                 3               4             4   
2                0.56                 4               2             4   
3                1.51                 3               6             1   
4                0.33                 6               5             4   
...               ...               ...             ...           ...   
53935            1.02                 2               4             2   
53936            0.93                 2               4             3   
53937            0.30                 4               5             4   
53938            0.36                 3               2             4   
53939            0.70                 1               2             2   

       features/depth  features/table  features/x  features/y  features/z  \
0           60.599998            60.0        6

In [11]:
df['label'].value_counts()

label
0    26985
1    26955
Name: count, dtype: int64

In [12]:
exact_size = 16500

In [13]:
cur_ratio = ratios[4]
print(cur_ratio)

(5, 5)


In [14]:
major, minor = cur_ratio

In [15]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [16]:
t0 = int(exact_size * major / (major + minor))
t1 = exact_size - t0 
print(t0,t1)

8250 8250


In [17]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [18]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [19]:
print(df)
print(df["label"].value_counts())

       features/carat  features/clarity  features/color  features/cut  \
0                0.56                 5               3             2   
1                0.33                 7               3             4   
2                1.19                 2               5             3   
3                1.35                 2               6             2   
4                1.02                 2               2             2   
...               ...               ...             ...           ...   
16495            1.34                 7               1             3   
16496            1.01                 1               5             0   
16497            0.38                 2               6             4   
16498            0.38                 2               3             4   
16499            1.01                 1               1             3   

       features/depth  features/table  features/x  features/y  features/z  \
0           61.400002            57.0        5

In [20]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [21]:
print(df_train_pool.head())
print(df_test.head())

   features/carat  features/clarity  features/color  features/cut  \
0            0.56                 5               3             2   
1            0.33                 7               3             4   
2            1.19                 2               5             3   
3            1.35                 2               6             2   
4            1.02                 2               2             2   

   features/depth  features/table  features/x  features/y  features/z  label  \
0       61.400002            57.0        5.26        5.32        3.25      0   
1       61.700001            55.0        4.47        4.45        2.75      0   
2       62.200001            58.0        6.73        6.77        4.20      1   
3       61.099998            61.0        7.10        7.13        4.35      1   
4       59.799999            58.0        6.49        6.55        3.90      1   

      id  
0  16508  
1   1783  
2  35552  
3  44044  
4  33617  
   features/carat  features/clarity  f

4. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes.

In [22]:
nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]

In [23]:
train_df = nested_train_dfs[7]

In [24]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[5.6000e-01 5.0000e+00 3.0000e+00 ... 5.3200e+00 3.2500e+00 1.6508e-06]
 [3.3000e-01 7.0000e+00 3.0000e+00 ... 4.4500e+00 2.7500e+00 1.7830e-07]
 [1.1900e+00 2.0000e+00 5.0000e+00 ... 6.7700e+00 4.2000e+00 3.5552e-06]
 ...
 [3.0000e-01 2.0000e+00 1.0000e+00 ... 4.2800e+00 2.7200e+00 5.3008e-06]
 [7.0000e-01 3.0000e+00 5.0000e+00 ... 5.7200e+00 3.5500e+00 3.9702e-06]
 [3.8000e-01 5.0000e+00 3.0000e+00 ... 4.6700e+00 2.9000e+00 3.5662e-06]]


In [25]:
test_df = df_test
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[7.0000e-01 3.0000e+00 2.0000e+00 ... 5.7200e+00 3.5400e+00 1.2530e-06]
 [9.1000e-01 1.0000e+00 2.0000e+00 ... 6.2300e+00 3.8200e+00 2.6582e-06]
 [1.0000e+00 5.0000e+00 1.0000e+00 ... 6.4200e+00 3.9700e+00 1.7877e-06]
 ...
 [3.8000e-01 2.0000e+00 6.0000e+00 ... 4.7000e+00 2.9000e+00 3.1090e-07]
 [3.8000e-01 2.0000e+00 3.0000e+00 ... 4.6800e+00 2.8600e+00 3.8341e-06]
 [1.0100e+00 1.0000e+00 1.0000e+00 ... 6.3900e+00 3.9900e+00 2.5028e-06]]


5 (Optional) The following code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [26]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

In [27]:
# from sklearn.metrics import pairwise_distances
# from sklearn.manifold import MDS
# import seaborn as sns
# import matplotlib.pyplot as plt

In [28]:
# D = pairwise_distances(X_all) 

In [29]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [30]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

6. Now we have the train_ds and test_ds for training

In [31]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

In [32]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [33]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  model_list.append(InfluenceModel(model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.6374 - accuracy: 0.6339 - val_loss: 0.5843 - val_accuracy: 0.7220 - 590ms/epoch - 18ms/step
32/32 - 0s - loss: 0.5557 - accuracy: 0.7561 - val_loss: 0.5278 - val_accuracy: 0.8260 - 76ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4894 - accuracy: 0.8330 - val_loss: 0.4400 - val_accuracy: 0.8800 - 77ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4268 - accuracy: 0.8717 - val_loss: 0.3843 - val_accuracy: 0.8900 - 91ms/epoch - 3ms/step
32/32 - 0s - loss: 0.3713 - accuracy: 0.8924 - val_loss: 0.3317 - val_accuracy: 0.8980 - 68ms/epoch - 2ms/step
32/32 - 0s - loss: 0.3237 - accuracy: 0.9062 - val_loss: 0.2949 - val_accuracy: 0.9020 - 71ms/epoch - 2ms/step
32/32 - 0s - loss: 0.2836 - accuracy: 0.9178 - val_loss: 0.2648 - val_accuracy: 0.9060 - 70ms/epoch - 2ms/step
32/32 - 0s - loss: 0.2502 - accuracy: 0.9280 - val_loss: 0.2398 - val_accuracy: 0.9080 - 66ms/epoch - 2ms/step
32/32 - 0s - loss: 0.2229 - accuracy: 0.9377 - val_loss: 0.2124 - val_accuracy: 0.9400 - 72ms/epoch - 2ms/step

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [34]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [35]:
start_if = time.perf_counter()

In [36]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0        16508  0.001481
1         1783  0.000181
2        35552  0.000019
3        44044  0.000002
4        33617  0.000145
...        ...       ...
7995     33279  0.000459
7996      5478  0.000102
7997     53008  0.000017
7998     39702  0.003624
7999     35662  0.000226

[8000 rows x 2 columns]


In [37]:
end_if = time.perf_counter()
runtime_if = end_if - start_if
print(f"FOIF Runtime: {runtime_if:.4f} seconds")

FOIF Runtime: 147.5075 seconds


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [38]:
start_tc = time.perf_counter()

In [39]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID         Score
0        16508  1.264778e-05
1         1783  3.448408e-07
2        35552 -4.161683e-09
3        44044 -1.971031e-09
4        33617 -2.632878e-08
...        ...           ...
7995     33279  2.932611e-05
7996      5478 -1.766280e-08
7997     53008  2.580297e-08
7998     39702  5.138377e-05
7999     35662  3.904956e-07

[8000 rows x 2 columns]


In [40]:
end_tc = time.perf_counter()
runtime_tc = end_tc - start_tc
print(f"TracIn Runtime: {runtime_tc:.4f} seconds")

TracIn Runtime: 294.8793 seconds


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [41]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [42]:
TracIn_sorted.to_csv("TC_Train_Set_Diamonds_Neighbor.csv",index = False)
df_sorted.to_csv("IF_Train_Set_Diamonds_Neighbor.csv",index = False)